# Goalkeeper Save Rate Analysis: Advanced vs Eliminated Teams (FIFA World Cup 2026)

**Analytic Question:** In the 2026 FIFA World Cup, do goalkeepers from teams that advanced to the knockout stage (Round of 32 or further) have a significantly higher save rate than goalkeepers from teams eliminated in the group stage?

**Data granularity:** Goalkeeper-match level (one row = one goalkeeper's performance in one match), **215 rows** in total, covering all 48 teams and 62 goalkeepers who appeared in this World Cup.

**Data source:** `data/gk_match.csv` (committed in this repository). Collected manually from FBref's per-goalkeeper Match Logs pages, filtered to `Comp == World Cup` rows, with each goalkeeper's totals (GA, SoTA, Saves, Minutes) cross-checked against the official season summary stats before merging. FBref's automated/bulk endpoints return HTTP 403 (Cloudflare bot protection) — this is a different, separate data source from the FIFA `api.fifa.com` + Playwright pipeline documented in the project README, which the rest of the team uses for `data/clean/team_match.csv`.


## Part 1: Load the data

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
import os

print(f"matplotlib version: {matplotlib.__version__}")

df = pd.read_csv("../data/gk_match.csv")
df['save_pct'] = pd.to_numeric(df['save_pct'], errors='coerce')

print(f"Total rows (goalkeeper-match): {len(df)}")
df.head()


matplotlib version: 3.8.4
Total rows (goalkeeper-match): 215


,date,round,Player,Team,opponent,minutes,shots_on_target_against,goals_against,saves,save_pct,advanced_to_knockout
0,2026-06-16,Group stage,Yazeed Abulaila,Jordan,at Austria,90,4.0,3,1.0,25.0,False
1,2026-06-22,Group stage,Yazeed Abulaila,Jordan,dz Algeria,90,8.0,2,6.0,75.0,False
2,2026-06-27,Group stage,Yazeed Abulaila,Jordan,ar Argentina,90,4.0,3,1.0,25.0,False
3,2026-06-13,Group stage,Mahmoud Abunada,Qatar,ch Switzerland,90,6.0,1,5.0,83.3,False
4,2026-06-18,Group stage,Mahmoud Abunada,Qatar,ca Canada,90,10.0,6,4.0,40.0,False


In [2]:
# In a small number of matches the goalkeeper faced zero shots on target (SoTA=0),
# making the save rate undefined (0/0). Pandas reads these as NaN, so we drop them
# before running the statistical analysis.
df_clean = df.dropna(subset=['save_pct'])
print(f"After removing rows where save rate is undefined (SoTA=0): {len(df_clean)}")

adv = df_clean.loc[df_clean['advanced_to_knockout'] == True, 'save_pct']
elim = df_clean.loc[df_clean['advanced_to_knockout'] == False, 'save_pct']

print(f"Advanced group sample size  n_A = {len(adv)}")
print(f"Eliminated group sample size n_B = {len(elim)}")


After removing rows where save rate is undefined (SoTA=0): 201
Advanced group sample size  n_A = 152
Eliminated group sample size n_B = 49


## Part 2: Descriptive statistics

In [3]:
def describe(series, label):
    print(f"--- {label} descriptive statistics ---")
    print(f"Mean:   {series.mean():.2f}")
    print(f"Median: {series.median():.2f}")
    print(f"Std:    {series.std(ddof=1):.2f}")
    print(f"Min:    {series.min():.1f}")
    print(f"Max:    {series.max():.1f}\n")

describe(adv, "Advanced group")
describe(elim, "Eliminated group")


--- Advanced group descriptive statistics ---
Mean:   67.36
Median: 66.70
Std:    28.57
Min:    0.0
Max:    100.0

--- Eliminated group descriptive statistics ---
Mean:   55.74
Median: 50.00
Std:    23.07
Min:    0.0
Max:    100.0



In [4]:
# NOTE: `labels=` (not `tick_labels=`) is used for boxplot category labels.
# `tick_labels=` was only added in matplotlib 3.9; requirements.txt pins
# matplotlib==3.8.4 (the test06 environment), where `tick_labels=` raises a
# TypeError. `labels=` is supported by both old and new matplotlib versions.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].boxplot([adv, elim], labels=[f"Advanced (n={len(adv)})", f"Eliminated (n={len(elim)})"])
axes[0].set_title("GK Save% by Team Outcome — Match-Level (2026 WC)")
axes[0].set_ylabel("Save %")

axes[1].hist(adv, bins=10, alpha=0.6, label="Advanced")
axes[1].hist(elim, bins=10, alpha=0.6, label="Eliminated")
axes[1].set_title("Distribution of Save%")
axes[1].set_xlabel("Save %")
axes[1].legend()

plt.tight_layout()

# Save into the shared figures/ folder (see project layout in README.md),
# matching the convention used by the other three notebooks.
FIGURES_DIR = "../figures"
os.makedirs(FIGURES_DIR, exist_ok=True)
fig_path = os.path.join(FIGURES_DIR, "goalkeeper_save_pct.png")
plt.savefig(fig_path, dpi=150)
print(f"Figure saved to: {fig_path}")

plt.show()


Figure saved to: ../figures/goalkeeper_save_pct.png


<Figure size 1100x450 with 2 Axes>

## Part 3: Confidence Interval (95% CI)

In [5]:
mean_adv = adv.mean()
sem_adv = stats.sem(adv)
ci_adv = stats.t.interval(confidence=0.95, df=len(adv) - 1, loc=mean_adv, scale=sem_adv)
print(f"Advanced group mean save rate 95% CI: {mean_adv:.2f} -> [{ci_adv[0]:.2f}, {ci_adv[1]:.2f}]")

mean_elim = elim.mean()
sem_elim = stats.sem(elim)
ci_elim = stats.t.interval(confidence=0.95, df=len(elim) - 1, loc=mean_elim, scale=sem_elim)
print(f"Eliminated group mean save rate 95% CI: {mean_elim:.2f} -> [{ci_elim[0]:.2f}, {ci_elim[1]:.2f}]")


Advanced group mean save rate 95% CI: 67.36 -> [62.78, 71.94]
Eliminated group mean save rate 95% CI: 55.74 -> [49.11, 62.37]


## Part 4: Assumption checks (normality & equal variance)

In [6]:
shapiro_adv = stats.shapiro(adv)
shapiro_elim = stats.shapiro(elim)
print(f"Shapiro-Wilk (Advanced):   W={shapiro_adv.statistic:.3f}, p={shapiro_adv.pvalue:.4f}")
print(f"Shapiro-Wilk (Eliminated): W={shapiro_elim.statistic:.3f}, p={shapiro_elim.pvalue:.4f}")

levene = stats.levene(adv, elim)
print(f"\nLevene's test for equal variances: statistic={levene.statistic:.3f}, p={levene.pvalue:.4f}")


Shapiro-Wilk (Advanced):   W=0.894, p=0.0000
Shapiro-Wilk (Eliminated): W=0.975, p=0.3751

Levene's test for equal variances: statistic=3.078, p=0.0809


**Normality is violated for the Advanced group** (Shapiro-Wilk p < 0.001), so the data cannot be assumed
to be normally distributed. This is expected here: save rate is a percentage bounded at 0 and 100, several
goalkeepers in the Advanced group posted a perfect 100% save rate in low-shot-volume matches, and the
distribution is visibly left-skewed in the histogram above — all of this pulls Shapiro-Wilk's p-value down
sharply.

Levene's test (p = 0.0809) indicates the two groups' variances are not significantly different, so unequal
variance is not an additional concern here — the normality violation is the main assumption at issue.

**Why we still proceed with a t-test, and how we check robustness:** with n_A ≈ 150 and n_B ≈ 49, the Central
Limit Theorem means the *sampling distribution of the mean* (which is what the t-test actually relies on) is
still reasonably close to normal even though the *raw data* is not — Welch's t-test is commonly considered
reasonably robust to non-normality at these sample sizes. To confirm the conclusion does not depend on this
assumption, we cross-check the result with a non-parametric **Mann-Whitney U test**, which makes no
distributional assumption at all, in the next section.

## Part 5: Two-sample t-test (Welch's t-test)

In [7]:
t_stat, p_value = stats.ttest_ind(adv, elim, equal_var=False, alternative="greater")
# alternative="greater" corresponds to the one-tailed hypothesis H1: mean(Advanced) > mean(Eliminated)

print("H0: mu_Advanced = mu_Eliminated")
print("H1: mu_Advanced > mu_Eliminated  (one-tailed)")
print(f"t statistic = {t_stat:.3f}")
print(f"p value     = {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"\nConclusion: p < {alpha}, reject H0. There is statistical evidence that goalkeepers "
          f"from advanced teams have a significantly higher save rate.")
else:
    print(f"\nConclusion: p >= {alpha}, fail to reject H0. There is insufficient evidence of a "
          f"significant difference between the two groups.")

pooled_std = np.sqrt(((len(adv) - 1) * adv.std(ddof=1) ** 2 +
                       (len(elim) - 1) * elim.std(ddof=1) ** 2) /
                      (len(adv) + len(elim) - 2))
cohens_d = (mean_adv - mean_elim) / pooled_std
print(f"\nEffect size Cohen's d = {cohens_d:.3f}")


H0: mu_Advanced = mu_Eliminated
H1: mu_Advanced > mu_Eliminated  (one-tailed)
t statistic = 2.885
p value     = 0.0024

Conclusion: p < 0.05, reject H0. There is statistical evidence that goalkeepers from advanced teams have a significantly higher save rate.

Effect size Cohen's d = 0.425


### Robustness check: Mann-Whitney U test (non-parametric, distribution-free)

In [8]:
u_stat, u_pvalue = stats.mannwhitneyu(adv, elim, alternative="greater")

print("Mann-Whitney U test (one-tailed, H1: Advanced group tends to have higher save rate)")
print(f"U statistic = {u_stat:.1f}")
print(f"p value     = {u_pvalue:.4f}")

if u_pvalue < alpha:
    print(f"\nConclusion: consistent with the t-test above — p < {alpha}, reject H0.")
else:
    print(f"\nConclusion: this contradicts the t-test above (p >= {alpha}); "
          f"the difference should be treated with more caution given the normality violation.")


Mann-Whitney U test (one-tailed, H1: Advanced group tends to have higher save rate)
U statistic = 4787.5
p value     = 0.0012

Conclusion: consistent with the t-test above — p < 0.05, reject H0.


Both tests agree: Welch's t-test (p = 0.0024) and the distribution-free Mann-Whitney U test point to the
same conclusion, which gives confidence that the significant result is not an artefact of the normality
violation identified in Part 4.

## Limitations

- Save percentage is heavily influenced by the quality and difficulty of shots faced, and is not a pure measure of a goalkeeper's individual skill.
- Some matches involved a goalkeeper playing only a few minutes (e.g., a late substitution), facing very few shots on target; single-match data for these cases can be noisy and add variance to the overall distribution.
- The Advanced group's save-rate distribution is not normally distributed (Shapiro-Wilk p < 0.001); this was cross-checked with a non-parametric Mann-Whitney U test, which agreed with the t-test conclusion, but the underlying skew should still be kept in mind when interpreting the mean/CI.
- Data collection was manual (not automated batch scraping). Although each goalkeeper's totals were cross-checked against official season summary statistics, manual compilation still carries some risk of human transcription error.
